# Реальная лабораторная 02 — Bike Sharing и Energy

Два временных датасета: городской спрос и сенсорные измерения.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## Часть A. Bike Sharing — почасовой спрос

In [ ]:
bike = load_real("bike_sharing")
print(bike.shape); display(bike.head()); display(bike.describe(include="all").T.head(30))

### Проверка календаря и целевой переменной

In [ ]:
bike.columns = [str(c).strip() for c in bike.columns]
date_col = next((c for c in bike.columns if c.lower() in {"dteday","date"}), None)
if date_col: bike[date_col] = pd.to_datetime(bike[date_col], errors="coerce")
for c in ["cnt","casual","registered","hr","temp","atemp","hum","windspeed"]:
    if c in bike: bike[c] = pd.to_numeric(bike[c], errors="coerce")
if "cnt" not in bike and {"casual","registered"}.issubset(bike.columns):
    bike["cnt"] = bike["casual"] + bike["registered"]
assert "cnt" in bike, "Не найден target cnt"
print("Период:", bike[date_col].min(), "—", bike[date_col].max() if date_col else "date unavailable")
print("Missing:", bike.isna().sum().sort_values(ascending=False).head())
if {"casual","registered"}.issubset(bike.columns):
    print("Проверка identity casual + registered = cnt:", np.allclose(bike["casual"]+bike["registered"], bike["cnt"], equal_nan=True))

### Суточные и сезонные паттерны

In [ ]:
if date_col and "hr" in bike:
    bike["timestamp"] = bike[date_col] + pd.to_timedelta(bike["hr"], unit="h")
    hourly = bike.sort_values("timestamp").set_index("timestamp")["cnt"]
    hourly.resample("D").sum().plot(figsize=(13,4), title="Daily total bike rentals"); plt.show()
if "hr" in bike:
    plt.figure(figsize=(10,4)); sns.lineplot(data=bike, x="hr", y="cnt", hue="workingday" if "workingday" in bike else None, errorbar=("ci",95))
    plt.title("Hourly demand profile"); plt.show()
if "season" in bike:
    display(bike.groupby("season")["cnt"].agg(["count","mean","median","std"]))

### Leakage и план прогнозирования

In [ ]:
leakage_candidates = [c for c in ["casual","registered","instant"] if c in bike.columns]
print("Требуют исключения/обоснования:", leakage_candidates)
if {"casual","registered","cnt"}.issubset(bike.columns):
    print("cnt - casual - registered max abs:", (bike["cnt"]-bike["casual"]-bike["registered"]).abs().max())
# Временной split должен идти по timestamp, а не случайным образом.
if "timestamp" in bike:
    cutoff = bike["timestamp"].quantile(.8)
    train, test = bike[bike["timestamp"] <= cutoff], bike[bike["timestamp"] > cutoff]
    print("Chronological split:", train.shape, test.shape, "cutoff=", cutoff)

## Часть B. Appliances Energy Prediction — сенсоры и энергопотребление

In [ ]:
energy = load_real("appliances_energy")
print(energy.shape); display(energy.head()); print(energy.dtypes.value_counts())

### Временной EDA и контрольные случайные признаки

In [ ]:
energy.columns = [str(c).strip() for c in energy.columns]
date_col_e = next((c for c in energy.columns if c.lower()=="date"), None)
if date_col_e: energy[date_col_e] = pd.to_datetime(energy[date_col_e], errors="coerce")
target_e = next((c for c in energy.columns if c.lower()=="appliances"), None)
assert target_e, "Не найден target Appliances"
print("Time range:", energy[date_col_e].min(), energy[date_col_e].max())
energy = energy.sort_values(date_col_e).set_index(date_col_e)
energy[target_e].resample("D").sum().plot(figsize=(13,4), title="Daily appliance energy (sum of 10-min readings)"); plt.show()
random_cols = [c for c in energy.columns if c.lower() in {"rv1","rv2"}]
print("Correlation with target (including random-control variables):")
display(energy.corr(numeric_only=True)[target_e].sort_values(ascending=False).to_frame("corr_with_Appliances"))

## Самостоятельные задания

1. Сравни временной профиль спроса велосипедов в рабочие и выходные дни.
2. Проверь, насколько случайное разбиение завышает качество относительно временного.
3. Для energy изучи автокорреляцию target и признаков сенсоров.
4. Проверь, дают ли rv1/rv2 заметную связь с Appliances в конкретной выборке; объясни случайные корреляции.
5. Составь таблицу «признак — доступен к моменту прогноза? — риск leakage».

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.